In [1]:
# =============================================================================
# MULTI-HORIZON LIGHTGBM WAVE FORECASTING — KBS PAPER  v3 (SPEED-OPTIMISED)
# Target: Colab Free Tier (T4 GPU, ~12GB RAM, ~90min session)
# Horizons: [3, 6, 12, 24h] | GPU | Optuna | Gap-CV | Full Checkpointing
#
# SPEED IMPROVEMENTS vs v2 (without sacrificing scientific rigour):
#   1. Feature pre-selection  : 722 → ≤200 features (4x fewer columns)
#   2. HPO on 50% data subset : same distribution, ~2x faster per trial
#   3. N_SPLITS 3→2 for HPO   : 33% fewer CV fits
#   4. N_TRIALS  20→12        : 40% fewer trials (TPE needs fewer with good prior)
#   5. Early stopping 50→30   : avoids over-growing trees during HPO
#   6. Tighter n_estimators   : 300–1200 (prior from run 1 showed ~1760 but
#                               on 50% data the effective optimum shifts lower)
#   7. float32 everywhere     : halves RAM for feature matrices
#   8. Three-layer checkpoint  : sentinel JSON + parquet shards + Optuna SQLite
# =============================================================================

!pip install -q lightgbm optuna shap pyarrow

import warnings, gc, pickle, json
warnings.filterwarnings("ignore")

import numpy             as np
import pandas            as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates  as mdates
import seaborn           as sns
import shap, optuna, lightgbm as lgb

from pathlib              import Path
from datetime             import datetime
from scipy.stats          import gaussian_kde
from sklearn.metrics      import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.samplers      import TPESampler

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── 1. PATHS ─────────────────────────────────────────────────────────────────
INPUT_DIR  = Path("/content/drive/MyDrive/KBS_Paper/Outputs/2_Feature_Engineering_KBS/")
OUTPUT_DIR = Path("/content/drive/MyDrive/KBS_Paper/Outputs/4_LightGBM_KBS/")
CKPT_DIR   = OUTPUT_DIR / "_checkpoints"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True,   exist_ok=True)

# ── 2. TUNABLE CONSTANTS ─────────────────────────────────────────────────────
HORIZONS        = [3, 6, 12, 24]
TIME_RES        = 3
TARGET_COL      = "target_buoy_hs"
RANDOM_STATE    = 42
VIS_HORIZON     = 6
VIS_WINDOW      = 14

# CV & HPO — tuned for free-tier speed
N_SPLITS_HPO    = 2          # 2 folds during HPO (33% faster vs 3)
N_SPLITS_FINAL  = 3          # 3 folds for final reported CV score (optional)
GAP             = 8          # 24h anti-leakage purge (fixed)
N_TRIALS        = 12         # TPE converges well in 12 trials (vs 20)
HPO_SUBSAMPLE   = 0.50       # fraction of train rows used during HPO trials
EARLY_STOP      = 30         # early-stopping patience during HPO
MAX_FEATURES    = 200        # cap after pre-selection step

# LightGBM HPO search bounds
LGB_N_EST_LO    = 300
LGB_N_EST_HI    = 1200
LGB_LR_LO       = 0.02
LGB_LR_HI       = 0.10

# ── 3. HELPERS ────────────────────────────────────────────────────────────────
def ckpt_path(h):   return CKPT_DIR / f"checkpoint_done_{h}h.json"
def shard_path(h, split): return CKPT_DIR / f"preds_{split}_{h}h.parquet"
def model_path(h):  return OUTPUT_DIR / f"lightgbm_model_{h}h.pkl"
def study_db(h):    return f"sqlite:///{CKPT_DIR}/optuna_{h}h.db"
def is_done(h):     return ckpt_path(h).exists()

def mark_done(h, meta):
    meta["completed_at"] = datetime.utcnow().isoformat()
    meta["horizon_hours"] = h
    ckpt_path(h).write_text(json.dumps(meta, indent=2))
    print(f"  ✓ Checkpoint saved → {ckpt_path(h).name}")

def load_meta(h):
    return json.loads(ckpt_path(h).read_text())

def mape(y_true, y_pred):
    m = y_true != 0
    return float(np.mean(np.abs((y_true[m] - y_pred[m]) / y_true[m])) * 100)

def metrics(y_true, y_pred):
    return {
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE" : float(mean_absolute_error(y_true, y_pred)),
        "R2"  : float(r2_score(y_true, y_pred)),
        "MAPE": mape(np.asarray(y_true, np.float32),
                     np.asarray(y_pred, np.float32)),
    }

def save_fig(fig, stem):
    for ext in ("png", "tiff"):
        fig.savefig(OUTPUT_DIR / f"{stem}.{ext}", dpi=600,
                    bbox_inches="tight", format=ext)
    print(f"  ✓ {stem}.png / .tiff")

# ── 4. DATA LOADING ───────────────────────────────────────────────────────────
print("=" * 68)
print("  LOADING DATA")
print("=" * 68)

def load(path):
    df = pd.read_csv(path)
    tc = next(c for c in df.columns if "time" in c.lower() or "date" in c.lower())
    df[tc] = pd.to_datetime(df[tc])
    df = df.rename(columns={tc: "time"}).set_index("time").sort_index()
    # Cast all numeric cols to float32 immediately — halves RAM
    df = df.astype({c: np.float32 for c in df.select_dtypes("number").columns})
    print(f"  {path.name:30s}  shape={df.shape}  RAM≈{df.memory_usage().sum()/1e6:.0f} MB")
    return df

df_train_raw = load(INPUT_DIR / "X_train_KBS.csv")
df_oos_raw   = load(INPUT_DIR / "X_oos_KBS.csv")

# ── 5. FAST FEATURE PRE-SELECTION ────────────────────────────────────────────
# Fit a shallow 100-tree LightGBM on the +3h target (the easiest horizon)
# and keep the top MAX_FEATURES by split gain importance.
# This single pass takes ~30-60s but saves 4-5 min per trial afterwards.

FEAT_CACHE = CKPT_DIR / "selected_features.json"

if FEAT_CACHE.exists():
    selected_features = json.loads(FEAT_CACHE.read_text())
    print(f"\n  Feature cache found — using {len(selected_features)} pre-selected features.")
else:
    print(f"\n  Running feature pre-selection (top {MAX_FEATURES} / {len(df_train_raw.columns)-1}) ...")
    _y_pre = df_train_raw[TARGET_COL].shift(-1).dropna()
    _X_pre = df_train_raw.drop(columns=[TARGET_COL]).loc[_y_pre.index]

    _pre_model = lgb.LGBMRegressor(
        n_estimators=100, num_leaves=31, max_depth=7,
        learning_rate=0.05, device="gpu",
        gpu_platform_id=0, gpu_device_id=0,
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1,
    )
    _pre_model.fit(_X_pre, _y_pre.values.astype(np.float32),
                   callbacks=[lgb.log_evaluation(-1)])

    importances = pd.Series(
        _pre_model.feature_importances_,
        index=_X_pre.columns
    ).sort_values(ascending=False)

    selected_features = importances.head(MAX_FEATURES).index.tolist()
    FEAT_CACHE.write_text(json.dumps(selected_features))

    print(f"  ✓ Selected {len(selected_features)} features  "
          f"(top importance: {selected_features[0]}, {selected_features[1]}, {selected_features[2]} ...)")
    del _pre_model, _X_pre, _y_pre, importances
    gc.collect()

# Slice both matrices to selected features only — done once, used everywhere
df_train = df_train_raw[selected_features + [TARGET_COL]]
df_oos   = df_oos_raw[selected_features + [TARGET_COL]]

# ── 6. HORIZON LOOP ───────────────────────────────────────────────────────────
already = [h for h in HORIZONS if is_done(h)]
todo    = [h for h in HORIZONS if not is_done(h)]

if already:
    print(f"\n  Checkpoints found — SKIPPING: {already}")
    for h in already:
        m = load_meta(h)
        print(f"    +{h}h  OOS R²={m.get('oos_R2', '?'):.4f}  @ {m.get('completed_at','')}")
if todo:
    print(f"  Horizons to train: {todo}\n")
else:
    print("  All horizons complete — jumping to assembly.\n")

for h in todo:
    step = int(h / TIME_RES)
    print("\n" + "=" * 68)
    print(f"  HORIZON: +{h}h  (step_shift={step})")
    print("=" * 68)

    # ── 6.1  Target alignment ─────────────────────────────────────────────────
    y_tr_s = df_train[TARGET_COL].shift(-step)
    y_os_s = df_oos[TARGET_COL].shift(-step)

    X_tr = df_train.drop(columns=[TARGET_COL]).loc[y_tr_s.notna()].astype(np.float32)
    y_tr = y_tr_s.dropna().values.astype(np.float32)
    ts_tr = X_tr.index

    X_os = df_oos.drop(columns=[TARGET_COL]).loc[y_os_s.notna()].astype(np.float32)
    y_os = y_os_s.dropna().values.astype(np.float32)
    ts_os = X_os.index

    print(f"  Train={len(X_tr):,}  OOS={len(X_os):,}  Features={X_tr.shape[1]}")

    # ── 6.2  HPO data subset (50% of train, time-contiguous tail) ────────────
    # Use the LAST 50% so it's the most recent / distribution-representative
    hpo_n   = int(len(X_tr) * HPO_SUBSAMPLE)
    X_hpo   = X_tr.iloc[-hpo_n:]
    y_hpo   = y_tr[-hpo_n:]
    tscv    = TimeSeriesSplit(n_splits=N_SPLITS_HPO, gap=GAP)

    # ── 6.3  Optuna objective ─────────────────────────────────────────────────
    def objective(trial):
        p = {
            "objective"        : "regression",
            "metric"           : "rmse",
            "verbosity"        : -1,
            "boosting_type"    : "gbdt",
            "device"           : "gpu",
            "gpu_platform_id"  : 0,
            "gpu_device_id"    : 0,
            "n_estimators"     : trial.suggest_int("n_estimators", LGB_N_EST_LO, LGB_N_EST_HI),
            "learning_rate"    : trial.suggest_float("learning_rate", LGB_LR_LO, LGB_LR_HI, log=True),
            "num_leaves"       : trial.suggest_int("num_leaves", 20, 80),
            "max_depth"        : trial.suggest_int("max_depth", 5, 12),
            "reg_alpha"        : trial.suggest_float("reg_alpha",  1e-4, 5.0, log=True),
            "reg_lambda"       : trial.suggest_float("reg_lambda", 1e-4, 5.0, log=True),
            "subsample"        : trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree" : trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 40),
            "random_state"     : RANDOM_STATE,
            "n_jobs"           : -1,
        }
        r2s = []
        for ti, vi in tscv.split(X_hpo):
            m = lgb.LGBMRegressor(**p)
            m.fit(X_hpo.iloc[ti], y_hpo[ti],
                  eval_set=[(X_hpo.iloc[vi], y_hpo[vi])],
                  callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False),
                             lgb.log_evaluation(-1)])
            r2s.append(r2_score(y_hpo[vi],
                                m.predict(X_hpo.iloc[vi],
                                          num_iteration=m.best_iteration_)))
        return float(np.mean(r2s))

    # ── 6.4  Run / resume Optuna study ───────────────────────────────────────
    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=RANDOM_STATE),
        storage=study_db(h),
        study_name=f"lgbm_{h}h",
        load_if_exists=True,
    )
    done_n  = sum(1 for t in study.trials
                  if t.state == optuna.trial.TrialState.COMPLETE)
    needed  = max(0, N_TRIALS - done_n)
    if needed == 0:
        print(f"  All {N_TRIALS} Optuna trials already complete — skipping HPO.")
    else:
        print(f"  {done_n}/{N_TRIALS} trials done — running {needed} more ...")
        study.optimize(objective, n_trials=needed, show_progress_bar=True)

    best_p    = study.best_params
    best_cv   = study.best_value
    print(f"\n  ✓ Best HPO CV R² = {best_cv:.4f}")
    print(f"  ✓ {best_p}")

    # ── 6.5  Final model on FULL train set ───────────────────────────────────
    print("\n  Retraining on full train set ...")
    final_p = {
        "objective"       : "regression",
        "metric"          : "rmse",
        "verbosity"       : -1,
        "boosting_type"   : "gbdt",
        "device"          : "gpu",
        "gpu_platform_id" : 0,
        "gpu_device_id"   : 0,
        "random_state"    : RANDOM_STATE,
        "n_jobs"          : -1,
        **best_p,
    }
    # Slightly boost n_estimators for full-data retrain (more data → more trees)
    final_p["n_estimators"] = min(int(best_p["n_estimators"] * 1.2), 1500)

    final_model = lgb.LGBMRegressor(**final_p)
    final_model.fit(X_tr, y_tr, callbacks=[lgb.log_evaluation(-1)])

    # ── 6.6  Evaluate ─────────────────────────────────────────────────────────
    pred_tr = final_model.predict(X_tr).astype(np.float32)
    pred_os = final_model.predict(X_os).astype(np.float32)
    mt = metrics(y_tr, pred_tr)
    mo = metrics(y_os, pred_os)
    print(f"  TRAIN | RMSE={mt['RMSE']:.4f}  R²={mt['R2']:.4f}  MAPE={mt['MAPE']:.2f}%")
    print(f"  OOS   | RMSE={mo['RMSE']:.4f}  R²={mo['R2']:.4f}  MAPE={mo['MAPE']:.2f}%")

    # ── 6.7  Save model ───────────────────────────────────────────────────────
    with open(model_path(h), "wb") as f:
        pickle.dump(final_model, f)

    # ── 6.8  Save prediction shards ───────────────────────────────────────────
    pd.DataFrame({"time": ts_tr, "split": "train", "horizon_hours": h,
                  "actual_hs": y_tr, "predicted_hs": pred_tr}
                 ).to_parquet(shard_path(h, "train"), index=False)
    pd.DataFrame({"time": ts_os, "split": "oos", "horizon_hours": h,
                  "actual_hs": y_os, "predicted_hs": pred_os}
                 ).to_parquet(shard_path(h, "oos"), index=False)

    # ── 6.9  Checkpoint sentinel ──────────────────────────────────────────────
    mark_done(h, {**{f"train_{k}": v for k, v in mt.items()},
                  **{f"oos_{k}": v for k, v in mo.items()},
                  "best_cv_r2": best_cv})

    del final_model, X_tr, X_os, pred_tr, pred_os
    gc.collect()

# ── 7. ASSEMBLE FINAL CSVs ────────────────────────────────────────────────────
print("\n" + "=" * 68)
print("  ASSEMBLING FINAL OUTPUTS")
print("=" * 68)

rows, tr_list, os_list = [], [], []
for h in HORIZONS:
    m  = load_meta(h)
    tr = pd.read_parquet(shard_path(h, "train"))
    os = pd.read_parquet(shard_path(h, "oos"))
    mt = metrics(tr["actual_hs"].values, tr["predicted_hs"].values)
    mo = metrics(os["actual_hs"].values, os["predicted_hs"].values)
    rows.append({"horizon_hours": h,
                 **{f"train_{k}": v for k, v in mt.items()},
                 **{f"oos_{k}":   v for k, v in mo.items()},
                 "best_cv_r2": m["best_cv_r2"]})
    tr_list.append(tr); os_list.append(os)

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(OUTPUT_DIR / "lightgbm_metrics_summary.csv", index=False)

train_all = pd.concat(tr_list, ignore_index=True)
oos_all   = pd.concat(os_list, ignore_index=True)
train_all.to_csv(OUTPUT_DIR / "lightgbm_train_predictions.csv", index=False)
oos_all.to_csv(  OUTPUT_DIR / "lightgbm_oos_predictions.csv",   index=False)

print(f"  ✓ lightgbm_metrics_summary.csv")
print(f"  ✓ lightgbm_train_predictions.csv  ({len(train_all):,} rows)")
print(f"  ✓ lightgbm_oos_predictions.csv    ({len(oos_all):,} rows)")
print("\n  OOS Metrics:")
print(metrics_df[["horizon_hours","oos_RMSE","oos_MAE","oos_R2","oos_MAPE"]].to_string(index=False))

# ── 8. Q1 VISUALISATIONS ─────────────────────────────────────────────────────
print("\n" + "=" * 68)
print(f"  Q1 VISUALISATIONS (+{VIS_HORIZON}h OOS)")
print("=" * 68)

sns.set_style("ticks")
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11,
                     "axes.titlesize": 13, "axes.labelsize": 12})

oos_6h = oos_all[oos_all["horizon_hours"] == VIS_HORIZON].copy()
oos_6h["time"] = pd.to_datetime(oos_6h["time"])
oos_6h = oos_6h.sort_values("time").reset_index(drop=True)
act, pred = oos_6h["actual_hs"].values, oos_6h["predicted_hs"].values
m6 = metrics(act, pred)

# ── 8.1  Density scatter ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6.5, 6.5))
xy  = np.vstack([act, pred])
kde = gaussian_kde(xy)(xy)
ix  = kde.argsort()
sc  = ax.scatter(act[ix], pred[ix], c=kde[ix], cmap="plasma",
                 s=12, alpha=0.75, linewidths=0, rasterized=True)
cbar = plt.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label("Kernel Density", fontsize=10)
lims = [min(act.min(), pred.min()) * 0.95, max(act.max(), pred.max()) * 1.05]
ax.plot(lims, lims, "k--", lw=1.5, label="1:1 Line")
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("Observed $H_s$ (m)"); ax.set_ylabel("Predicted $H_s$ (m)")
ax.set_title(f"LightGBM +{VIS_HORIZON}h  |  OOS\n"
             f"RMSE={m6['RMSE']:.3f} m  MAE={m6['MAE']:.3f} m  "
             f"$R^2$={m6['R2']:.3f}  MAPE={m6['MAPE']:.1f}%")
ax.legend(fontsize=10); sns.despine(fig=fig)
save_fig(fig, "fig1_density_scatter_6h_oos"); plt.close(fig)

# ── 8.2  Storm-window time series ────────────────────────────────────────────
peak_t = oos_6h.loc[oos_6h["actual_hs"].idxmax(), "time"]
half   = pd.Timedelta(days=VIS_WINDOW / 2)
win    = oos_6h[(oos_6h["time"] >= peak_t - half) & (oos_6h["time"] <= peak_t + half)]

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(win["time"], win["actual_hs"],    lw=1.8, color="#1f77b4", label="Observed $H_s$")
ax.plot(win["time"], win["predicted_hs"], lw=1.5, color="#d62728",
        ls="--", label=f"LightGBM +{VIS_HORIZON}h")
ax.axvline(peak_t, color="k", lw=1, ls=":", label=f"Peak ({peak_t.date()})")
ax.set_xlabel("Date (UTC)"); ax.set_ylabel("$H_s$ (m)")
ax.set_title(f"LightGBM +{VIS_HORIZON}h — Storm window ±{VIS_WINDOW//2} days (OOS)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
ax.legend(fontsize=10); sns.despine(fig=fig); fig.tight_layout()
save_fig(fig, "fig2_timeseries_storm_6h_oos"); plt.close(fig)

# ── 8.3  SHAP (top 15) ───────────────────────────────────────────────────────
with open(model_path(VIS_HORIZON), "rb") as f:
    model_6h = pickle.load(f)

step_6h = int(VIS_HORIZON / TIME_RES)
y_6h_s  = df_oos[TARGET_COL].shift(-step_6h)
X_6h_os = df_oos.drop(columns=[TARGET_COL]).loc[y_6h_s.notna()].astype(np.float32)

rng     = np.random.default_rng(RANDOM_STATE)
sidx    = rng.choice(len(X_6h_os), min(800, len(X_6h_os)), replace=False)
X_shap  = X_6h_os.iloc[sidx]

explainer  = shap.TreeExplainer(model_6h)
shap_vals  = explainer.shap_values(X_shap)
mean_abs   = np.abs(shap_vals).mean(axis=0)
top15      = np.argsort(mean_abs)[::-1][:15]
names15    = [X_6h_os.columns[i] for i in top15]
vals15     = mean_abs[top15]

fig, ax = plt.subplots(figsize=(8, 5.5))
clrs    = plt.cm.RdBu_r(np.linspace(0.15, 0.85, 15))[::-1]
bars    = ax.barh(range(15)[::-1], vals15, color=clrs, edgecolor="white", lw=0.4)
ax.set_yticks(range(15)[::-1]); ax.set_yticklabels(names15, fontsize=9)
ax.set_xlabel("Mean |SHAP| value")
ax.set_title(f"LightGBM +{VIS_HORIZON}h — Top 15 SHAP Features (OOS)")
for bar, val in zip(bars, vals15[::-1]):
    ax.text(val + 0.0005, bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", va="center", fontsize=7.5)
sns.despine(fig=fig); fig.tight_layout()
save_fig(fig, "fig3_shap_summary_6h_oos"); plt.close(fig)

pd.DataFrame(shap_vals, columns=X_6h_os.columns).to_csv(
    OUTPUT_DIR / "shap_values_6h_oos_sample.csv", index=False)
print("  ✓ shap_values_6h_oos_sample.csv")

# ── 9. MANIFEST ───────────────────────────────────────────────────────────────
print("\n" + "=" * 68)
print("  COMPLETE — FILE MANIFEST")
print("=" * 68)
for p in sorted(OUTPUT_DIR.glob("*")):
    if p.is_file():
        print(f"  {p.name:<55}  {p.stat().st_size/1024:>7.1f} KB")
print(f"\n  Output dir: {OUTPUT_DIR}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 7.5 MB/s eta 0:00:00
  LOADING DATA
  X_train_KBS.csv                 shape=(12477, 722)  RAM≈36 MB
  X_oos_KBS.csv                   shape=(5605, 722)  RAM≈16 MB

  Running feature pre-selection (top 200 / 721) ...
  ✓ Selected 200 features  (top importance: target_buoy_hs_roll_max_6h, target_buoy_windspeed, target_buoy_hs_roll_mean_6h ...)
  Horizons to train: [3, 6, 12, 24]


  HORIZON: +3h  (step_shift=1)
  Train=12,476  OOS=5,604  Features=200
  All 12 Optuna trials already complete — skipping HPO.

  ✓ Best HPO CV R² = 0.5219
  ✓ {'n_estimators': 1435, 'learning_rate': 0.02142387495644906, 'num_leaves': 25, 'max_depth': 8, 'reg_alpha': 0.0042258746449961694, 'reg_lambda': 0.44466289554754435, 'subsample': 0.8550229885420852, 'colsample_bytree': 0.9548850970305306, 'min_child_samples': 29}

  Retraining on full train set ...
  TRAIN | RMSE=0.0858  R²=0.9077  MAPE=12.54%
  OOS   | RMSE=0.2248  R²=0.4464  MAPE=22.34%
  ✓ Che

  0%|          | 0/12 [00:00<?, ?it/s]


  ✓ Best HPO CV R² = 0.4080
  ✓ {'n_estimators': 1159, 'learning_rate': 0.060621262929391274, 'num_leaves': 35, 'max_depth': 5, 'reg_alpha': 0.04547774761586353, 'reg_lambda': 0.005011121598680179, 'subsample': 0.7070730754849486, 'colsample_bytree': 0.756981156841053, 'min_child_samples': 30}

  Retraining on full train set ...
  TRAIN | RMSE=0.0696  R²=0.9394  MAPE=11.05%
  OOS   | RMSE=0.2400  R²=0.3689  MAPE=29.72%
  ✓ Checkpoint saved → checkpoint_done_6h.json

  HORIZON: +12h  (step_shift=4)
  Train=12,473  OOS=5,601  Features=200
  0/12 trials done — running 12 more ...


  0%|          | 0/12 [00:00<?, ?it/s]


  ✓ Best HPO CV R² = 0.2970
  ✓ {'n_estimators': 1080, 'learning_rate': 0.04360598649485911, 'num_leaves': 41, 'max_depth': 11, 'reg_alpha': 0.013925808515600444, 'reg_lambda': 3.173228241054352, 'subsample': 0.8890237037952976, 'colsample_bytree': 0.8293163910604917, 'min_child_samples': 40}

  Retraining on full train set ...
  TRAIN | RMSE=0.0347  R²=0.9849  MAPE=6.27%
  OOS   | RMSE=0.2526  R²=0.3011  MAPE=35.36%
  ✓ Checkpoint saved → checkpoint_done_12h.json

  HORIZON: +24h  (step_shift=8)
  Train=12,469  OOS=5,597  Features=200
  0/12 trials done — running 12 more ...


  0%|          | 0/12 [00:00<?, ?it/s]


  ✓ Best HPO CV R² = 0.1574
  ✓ {'n_estimators': 637, 'learning_rate': 0.09237421878009823, 'num_leaves': 64, 'max_depth': 9, 'reg_alpha': 0.0005409123677836541, 'reg_lambda': 0.0005407712220288125, 'subsample': 0.6232334448672797, 'colsample_bytree': 0.9464704583099741, 'min_child_samples': 28}

  Retraining on full train set ...
  TRAIN | RMSE=0.0114  R²=0.9984  MAPE=2.06%
  OOS   | RMSE=0.2748  R²=0.1731  MAPE=46.06%
  ✓ Checkpoint saved → checkpoint_done_24h.json

  ASSEMBLING FINAL OUTPUTS
  ✓ lightgbm_metrics_summary.csv
  ✓ lightgbm_train_predictions.csv  (49,893 rows)
  ✓ lightgbm_oos_predictions.csv    (22,405 rows)

  OOS Metrics:
 horizon_hours  oos_RMSE  oos_MAE   oos_R2  oos_MAPE
             3  0.224822 0.086478 0.446404 22.338251
             6  0.240045 0.110362 0.368892 29.718473
            12  0.252606 0.127332 0.301099 35.364632
            24  0.274764 0.155320 0.173074 46.058464

  Q1 VISUALISATIONS (+6h OOS)
  ✓ fig1_density_scatter_6h_oos.png / .tiff
  ✓ fig2_t